# Phase 5: Model Training & Metric Engineering

Before building the machine learning classifiers, I must establish exactly how I am going to evaluate them and how they will execute trades. Standard machine learning metrics are designed for academic datasets, not the financial markets. To build a commercial-grade trading bot, I have to abandon traditional accuracy and optimize strictly for risk management and capital preservation.

## 1. The Accuracy Paradox (Why I Cannot Use Accuracy)
In standard binary classification, the default metric is Accuracy. However, in trading—especially with setups that naturally have a ~35% baseline win rate to achieve a 1:2 Risk-to-Reward (R:R)—Accuracy is a dangerous trap. 

Because my datasets are naturally imbalanced (roughly 65% of the trades hit Stop Loss), a lazy machine learning model will quickly realize that the easiest way to minimize error is to simply predict "0" (Loss) for every single trade. The system will report a highly impressive 65% Accuracy, but the bot will never take a single trade. Accuracy gives the illusion of a working model while actively avoiding market participation. Therefore, Accuracy is completely useless to my system.

## 2. Mathematical Expectancy & Continuous Probability (`predict_proba`)
Instead of forcing the model to output a strict binary decision (`1` for Win, `0` for Loss) using `.predict()`, I will use `.predict_proba()`. This forces the algorithm to output an array of continuous probabilities (from 0.0 to 1.0) indicating exactly how confident it is in the setup.

This shift allows me to calculate and optimize for **Mathematical Expectancy**. Expectancy is the average amount of risk I expect to yield per trade, calculated as:

$$ \text{Expectancy} = (P(\text{Win}) \times \text{Reward}) - (P(\text{Loss}) \times \text{Risk}) $$

By knowing the exact probability curve of a setup, I can dynamically manage risk rather than blindly entering trades.

## 3. The 3-Zone Probability Architecture
Treating probabilities as a spectrum allows me to create a highly robust execution framework. Instead of treating a 51% loss probability as a signal to short, I have divided the model's confidence into three distinct zones, taking action in only two of them:

*   **Zone 1: The Long Trade (High Conviction Win):** If the model predicts a high probability of success (e.g., $P > 0.65$), I execute the standard setup targeting the normal 1:2 R:R. 
*   **Zone 2: The Neutral Zone (Cash is a Position):** If the probability falls in the middle (e.g., $0.20 \le P \le 0.65$), the model lacks overwhelming conviction in either direction. In this zone, I do nothing. Missing a trade costs absolutely nothing, and sitting in cash protects capital from noise.
*   **Zone 3: The Reverse Trade (High Conviction Loss):** If the model is overwhelmingly confident the setup is garbage (e.g., $P < 0.20$), I take the exact inverse of the trade. The risk profile flips from 1:2 to a 2:1 (risking 2 to make 1), but because the probability of the original setup failing is so massive, the mathematical expectancy remains highly positive.

I only expose capital in Zone 1 and Zone 3.

## 4. The Flaws of Precision and Recall
To optimize the model's ability to classify these zones, I must evaluate it correctly. However, standalone metrics have fatal blind spots in quantitative finance:

*   **Why Precision Fails Alone:** Precision measures how often the bot wins when it decides to trade. If I strictly optimize for 90% Precision, the model becomes terrified. It will only take the absolute most perfect setups, perhaps trading twice a year. While it wins those trades, it leaves massive amounts of capital on the table and completely fails to generate meaningful portfolio growth.
*   **Why Recall Fails Alone:** Recall measures how many of the actual winning trades the model successfully caught. If I optimize for 100% Recall, the model is forced to take almost every single trade to ensure it doesn't miss a winner. This drags the system's win rate right back down to the 35% baseline, guaranteeing long-term failure. 

## 5. The Ultimate Trading Metric: PR-AUC
Because Precision makes the model too cautious and Recall makes it too aggressive, I need a metric that perfectly balances the two. 

I will use **PR-AUC (Precision-Recall Area Under Curve)**. 

If I calculate the Precision and Recall at every possible probability threshold and plot it on a graph, the AUC calculates the mathematical area beneath that curve. 
Most tutorials recommend ROC-AUC, but ROC-AUC rewards a model for correctly identifying True Negatives (the losing trades it skipped). In an imbalanced trading dataset, a model can easily rack up True Negatives just by predicting "0" most of the time, inflating the score.

PR-AUC explicitly ignores True Negatives. It offers zero reward for correctly identifying garbage setups. It forces the machine learning algorithm to be judged entirely on one unforgiving standard: *How efficiently did it extract the rare, profitable trades out of the baseline noise?* By optimizing for PR-AUC, I guarantee the model is maximizing real profitability.

## Step 1: Baseline XGBoost Evaluation

I currently have 45 model-ready datasets of varying sizes. Running a full hyperparameter grid search on all of them would be a massive waste of computational resources, especially since many datasets might not contain enough signal to be viable in the first place. 

Drawing on my experience from the F1-PitWall project, I know that hyperparameter tuning (like depth, learning rate, and regularization) typically yields a 5% to 6% performance boost. Therefore, my strategy is to approach this in two stages:

1. **The Baseline Test:** I will first train an XGBoost classifier using strictly default settings across all datasets. I will separate the `Dataset_WinRate` column to use as my baseline benchmark, and train the model on the remaining scaled features.
2. **The Filter:** I will evaluate each dataset's performance using PR-AUC and base Precision (at a 0.50 threshold). If the default model cannot beat the raw baseline win rate, the dataset is dropped. 
3. **The Tuning Phase:** Once I know exactly which datasets actually have a mathematical edge and survived the baseline test, I will dedicate my compute power to running a grid search only on those survivors to squeeze out that final 5-6% alpha.

The code block below executes this baseline test and generates a survival report.

In [1]:
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, precision_score
from xgboost import XGBClassifier

ready_dir = '../data/model_ready/'
csv_files = glob.glob(os.path.join(ready_dir, '*.csv'))

survivors = []
summary_log = []

for file in csv_files:
    filename = os.path.basename(file)
    df = pd.read_csv(file)
          
    baseline_winrate = df['Dataset_WinRate'].iloc[0]
    
    X = df.drop(columns=['TP_or_SL', 'Dataset_WinRate'])
    y = df['TP_or_SL']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    
    model = XGBClassifier(random_state=42, eval_metric='logloss')
    model.fit(X_train, y_train)
    
    y_probs = model.predict_proba(X_test)[:, 1]
    
    pr_auc = average_precision_score(y_test, y_probs)
    
    y_pred_default = (y_probs >= 0.50).astype(int)
    precision = precision_score(y_test, y_pred_default, zero_division=0)
    
    improvement = precision - baseline_winrate
    
    if precision > baseline_winrate and pr_auc > baseline_winrate:
        status = "SURVIVED"
        survivors.append(filename)
    else:
        status = "FAILED"
        os.remove(file)
        
    summary_log.append(
        f"[{status}] {filename:<20} | Baseline: {baseline_winrate:.1%} | "
        f"PR-AUC: {pr_auc:.3f} | Precision: {precision:.1%} | Improv: {improvement:+.1%}"
    )

print("BASELINE XGBOOST EVALUATION REPORT")
print("-" * 85)
for log in sorted(summary_log):
    print(log)
print("-" * 85)
print(f"Total Survivors: {len(survivors)} / {len(csv_files)}")

BASELINE XGBOOST EVALUATION REPORT
-------------------------------------------------------------------------------------
[FAILED] fvg_1H_Nasdaq.csv    | Baseline: 39.6% | PR-AUC: 0.354 | Precision: 36.5% | Improv: -3.1%
[FAILED] fvg_1H_Silver.csv    | Baseline: 35.6% | PR-AUC: 0.309 | Precision: 30.4% | Improv: -5.2%
[FAILED] ob_15M_Gold.csv      | Baseline: 36.6% | PR-AUC: 0.339 | Precision: 35.6% | Improv: -1.0%
[FAILED] ob_1H_Nasdaq.csv     | Baseline: 39.9% | PR-AUC: 0.316 | Precision: 31.0% | Improv: -8.8%
[FAILED] ote_15M_EURUSD.csv   | Baseline: 31.9% | PR-AUC: 0.343 | Precision: 27.2% | Improv: -4.7%
[FAILED] ote_15M_Gold.csv     | Baseline: 32.2% | PR-AUC: 0.310 | Precision: 28.3% | Improv: -3.9%
[FAILED] ote_1H_EURUSD.csv    | Baseline: 26.7% | PR-AUC: 0.222 | Precision: 14.9% | Improv: -11.8%
[FAILED] ote_1H_Gold.csv      | Baseline: 27.4% | PR-AUC: 0.147 | Precision: 8.2% | Improv: -19.2%
[FAILED] ote_1H_Nasdaq.csv    | Baseline: 34.9% | PR-AUC: 0.307 | Precision: 11.1% | I

## Step 2: Hyperparameter Tuning (The Grid Search)

With the noise eliminated, I now have 33 structurally sound datasets that proved they hold a mathematical edge. My goal now is to extract the final 5-6% of performance through hyperparameter optimization, similar to the precision engineering required in high-level classification tasks. 

Financial data is notoriously prone to overfitting. If I allow the tree models to grow too deep, they will memorize the historical noise rather than the underlying institutional order flow. Therefore, I am designing a highly restricted, conservative grid search:

*   **Shallow Trees (`max_depth`: 3 to 5):** Forces the model to only split on the most undeniable macro-features.
*   **Subsampling (`subsample` & `colsample_bytree`: 0.8):** Randomly drops 20% of the data and features during each boosting round, preventing the model from relying too heavily on any single metric.
*   **Metric (`scoring`: average_precision):** The grid search will evaluate every combination strictly on PR-AUC.

The absolute best model for each dataset will be serialized and saved as a `.pkl` file in the `../models/` directory, ready to be loaded by the modular backtesting engine.

In [2]:
import os
import glob
import pandas as pd
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import joblib

ready_dir = '../data/model_ready/'
model_dir = '../models/'

os.makedirs(model_dir, exist_ok=True)

csv_files = glob.glob(os.path.join(ready_dir, '*.csv'))

param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

summary_log = []

print("STARTING HYPERPARAMETER GRID SEARCH")
print("=" * 80)

for file in csv_files:
    filename = os.path.basename(file)
    model_name = filename.replace('.csv', '_xgb.pkl')
    df = pd.read_csv(file)
    
    if df.empty:
        continue
        
    X = df.drop(columns=['TP_or_SL', 'Dataset_WinRate'])
    y = df['TP_or_SL']
    
    xgb = XGBClassifier(random_state=42, eval_metric='logloss')
    
    grid_search = GridSearchCV(
        estimator=xgb,
        param_grid=param_grid,
        scoring='average_precision',
        cv=3,
        n_jobs=-1,
        verbose=0
    )
    
    grid_search.fit(X, y)
    
    best_model = grid_search.best_estimator_
    best_pr_auc = grid_search.best_score_
    
    joblib.dump(best_model, os.path.join(model_dir, model_name))
    
    summary_log.append(f"[SAVED] {model_name:<25} | Best PR-AUC: {best_pr_auc:.3f} | Params: {grid_search.best_params_}")
    print(f"Tuned and saved: {model_name}")

print("\nGRID SEARCH COMPLETED")
print("=" * 80)
for log in sorted(summary_log):
    print(log)
print("=" * 80)
print(f"All optimized models saved to {model_dir}")

STARTING HYPERPARAMETER GRID SEARCH
Tuned and saved: fvg_15M_EURUSD_xgb.pkl
Tuned and saved: fvg_15M_Gold_xgb.pkl
Tuned and saved: fvg_15M_Nasdaq_xgb.pkl
Tuned and saved: fvg_15M_Silver_xgb.pkl
Tuned and saved: fvg_15M_SP500_xgb.pkl
Tuned and saved: fvg_1H_EURUSD_xgb.pkl
Tuned and saved: fvg_1H_Gold_xgb.pkl
Tuned and saved: fvg_1H_SP500_xgb.pkl
Tuned and saved: fvg_5M_EURUSD_xgb.pkl
Tuned and saved: fvg_5M_Gold_xgb.pkl
Tuned and saved: fvg_5M_Nasdaq_xgb.pkl
Tuned and saved: fvg_5M_Silver_xgb.pkl
Tuned and saved: fvg_5M_SP500_xgb.pkl
Tuned and saved: ob_15M_EURUSD_xgb.pkl
Tuned and saved: ob_15M_Nasdaq_xgb.pkl
Tuned and saved: ob_15M_Silver_xgb.pkl
Tuned and saved: ob_15M_SP500_xgb.pkl
Tuned and saved: ob_1H_EURUSD_xgb.pkl
Tuned and saved: ob_1H_Gold_xgb.pkl
Tuned and saved: ob_1H_Silver_xgb.pkl
Tuned and saved: ob_1H_SP500_xgb.pkl
Tuned and saved: ob_5M_EURUSD_xgb.pkl
Tuned and saved: ob_5M_Gold_xgb.pkl
Tuned and saved: ob_5M_Nasdaq_xgb.pkl
Tuned and saved: ob_5M_Silver_xgb.pkl
Tuned a